# commaVQ LoRA gate

Runtime → Change runtime type → **T4 GPU**, then Run all.

LoRA-finetunes gpt2m on commaVQ, then compares **held-out** bits/token of the finetuned model vs the frozen base. Prints SHIP / DO NOT SHIP. Downloads the adapter if it helps.

Gate: only ship the adapter if held-out cross-entropy drops enough to beat its size over the full 768M-token dataset.

In [ ]:
# No peft (its newest build needs torchao>=0.16, which Colab lacks). Manual LoRA below.
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation
!pip -q install datasets
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ---------------- CONFIG (fast + reliable for free Colab) ----------------
MODEL = 'commaai/commavq-gpt2m'
SPLIT_FILES = ['data-0000.tar.gz', 'data-0001.tar.gz']
TRAIN_SEGMENTS = 40
EVAL_SEGMENTS  = 4
STEPS = 600              # ~7 min at batch1 no-checkpointing; raise if session is stable
BATCH = 1
EVAL_BATCH = 2
LR = 5e-5
RANK = 16
CKPT = False            # batch1 fits T4 (~5GB) WITHOUT checkpointing -> ~3x faster. Set True only if OOM.
OUT = 'commavq_lora_adapter'
# -------------------------------------------------------------------------
import math, time, numpy as np, torch
BOS, TPF, GRID, BLOCK_FRAMES = 1024, 129, 128, 20
BLOCK = BLOCK_FRAMES * TPF  # 2580
def log(*a): print(*a, flush=True)

def load_segments(n):
    from datasets import load_dataset
    ds = load_dataset('commaai/commavq', data_files={'train': SPLIT_FILES})['train']
    return [np.array(ds[i]['token.npy']).reshape(-1, GRID).astype(np.int64) for i in range(min(n, ds.num_rows))]

def to_blocks(segs):
    blocks = []
    for t in segs:
        flat = np.concatenate([np.concatenate([[BOS], fr]) for fr in t])
        n = (flat.shape[0] // BLOCK) * BLOCK
        for i in range(0, n, BLOCK):
            blocks.append(flat[i:i+BLOCK])
    return np.stack(blocks) if blocks else np.zeros((0, BLOCK), np.int64)

def content_bits(model, blocks, device, batch=2):
    model.eval(); tot_nll = tot_n = 0
    with torch.no_grad():
        for i in range(0, len(blocks), batch):
            ids = torch.tensor(blocks[i:i+batch], dtype=torch.long, device=device)
            logits = model(ids).logits[:, :-1, :]
            tgt = ids[:, 1:]
            logp = torch.log_softmax(logits.float(), dim=-1)
            nll = -logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
            mask = (tgt != BOS)
            tot_nll += float(nll[mask].sum()); tot_n += int(mask.sum())
    return tot_nll / tot_n / math.log(2)

In [ ]:
import torch.nn as nn
from transformers import GPT2LMHeadModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'

segs = load_segments(TRAIN_SEGMENTS + EVAL_SEGMENTS)
train_blocks = to_blocks(segs[:TRAIN_SEGMENTS])
eval_blocks  = to_blocks(segs[TRAIN_SEGMENTS:])
log(f'train blocks={len(train_blocks)} eval blocks={len(eval_blocks)}')

model = GPT2LMHeadModel.from_pretrained(MODEL).to(device)
base_bits = content_bits(model, eval_blocks, device, EVAL_BATCH)
log(f'BASE held-out bits/token: {base_bits:.4f}  (ratio {10/base_bits:.3f})')

# ---- manual LoRA on ALL Conv1D layers (attn + mlp) -- no peft/torchao ----
for p in model.parameters():
    p.requires_grad_(False)

class LoRAConv1D(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        nx, nf = base.weight.shape
        dev = base.weight.device
        self.A = nn.Parameter(torch.randn(nx, r, device=dev) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, nf, device=dev))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A) @ self.B)

for blk in model.transformer.h:
    blk.attn.c_attn = LoRAConv1D(blk.attn.c_attn, RANK, 2 * RANK)
    blk.attn.c_proj = LoRAConv1D(blk.attn.c_proj, RANK, 2 * RANK)
    blk.mlp.c_fc    = LoRAConv1D(blk.mlp.c_fc,    RANK, 2 * RANK)
    blk.mlp.c_proj  = LoRAConv1D(blk.mlp.c_proj,  RANK, 2 * RANK)
model.to(device)

if CKPT:
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

lora_params = [p for p in model.parameters() if p.requires_grad]
ntrain = sum(p.numel() for p in lora_params)
log(f'trainable (LoRA) params: {ntrain}')
opt = torch.optim.AdamW(lora_params, lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
model.train(); rng = np.random.default_rng(0); t0 = time.time(); run = 0.0
for step in range(STEPS):
    idx = rng.integers(0, len(train_blocks), size=BATCH)
    ids = torch.tensor(train_blocks[idx], dtype=torch.long, device=device)
    out = model(ids, labels=ids); out.loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    run += out.loss.item()
    if (step + 1) % 50 == 0:
        log(f'  step {step+1}/{STEPS} avg-loss {run/50:.4f} ({(time.time()-t0)/(step+1):.2f}s/step)'); run = 0.0

# ---- GATE (same cell -> no ordering mistakes) ----
if CKPT:
    model.gradient_checkpointing_disable(); model.config.use_cache = True
ft_bits = content_bits(model, eval_blocks, device, EVAL_BATCH)
adapter_mb = ntrain * 2 / 1e6
gain = base_bits - ft_bits
saved_mb = gain * 768e6 / 8 / 1e6
log('=== LoRA GATE (held-out) ===')
log(f'base bits/token     : {base_bits:.4f}  (ratio {10/base_bits:.3f})')
log(f'finetuned bits/token: {ft_bits:.4f}  (ratio {10/ft_bits:.3f})')
log(f'gain                : {gain:.4f} bits/token ({100*gain/base_bits:.2f}%)')
log(f'adapter (~fp16)     : {adapter_mb:.2f} MB')
log(f'full-dataset bytes saved est: {saved_mb:.1f} MB  vs adapter {adapter_mb:.2f} MB')
log('VERDICT:', 'SHIP (net win)' if (saved_mb > adapter_mb and gain > 0) else 'DO NOT SHIP (no net gain)')
if gain > 0:
    sd = {n: p.detach().cpu().half() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(sd, OUT + '.pt')
    from google.colab import files; files.download(OUT + '.pt')